# 00 はじめに — 基礎知識と用語から始めよう（強化学習）

> 🔒 すべて**自作のシミュレータ**と**合成データ**です。⏱ 所要時間: 読む 30分 ＋ 実行 数秒。**難しければ、「図の見方」と「用語集」だけ読んで、次のNotebookへ進んでください。**

予備知識ゼロから始めます。数式はほとんど使いません。分からない言葉が出たら、ここに戻ってきてください。

## 1. このプロジェクトは何をするの？（ひとことで）
**「正解を教わらず、報酬（点数）だけを手がかりに、うまい行動を自分で見つける」AI**を作ります。これを**強化学習**といいます。

たとえ話: 犬に「お手」を教えるとき、手取り足取り教えるのではなく、**うまくできたらおやつをあげる**。犬は試行錯誤して、おやつが増える行動を覚えます。強化学習のAIも同じです。

迷路の攻略、ゲームのAI、ロボットの制御、電力の運用（このプロジェクトでは家庭用蓄電池）などに使われます。

## 2. 基礎知識

### 2-1. 他の学習との違い
| 学習 | 教えてもらうもの | 例 |
|---|---|---|
| **教師あり** | 入力と**正解** | 画像→「これは猫」 |
| **教師なし** | 正解なし。データの構造を見つける | 似たお客さんをグループ分け |
| **強化学習** | **報酬だけ**（行動のあとに、点数が返る） | 迷路を最短で抜ける |
強化学習では、「正しい行動」は教えられません。**やってみて、結果の点数から**学びます。しかも、**今の行動の結果が、ずっと先に現れる**（ゴールに着いて初めて点が分かる）ことがあります。

### 2-2. 5つの基本用語
繰り返しのやりとりは、次のとおりです。
```
  エージェント ──行動──▶ 環境
      ▲                    │
      └──状態・報酬────────┘
```
- **エージェント**: 学ぶ側（AI）。 **環境**: エージェントが行動する相手（迷路・ゲーム・家と蓄電池）。
- **状態**: 今の状況（迷路なら、いるマス）。 **行動**: できること（上・右・下・左）。 **報酬**: 行動の結果として返る点数。
- **方策**: 状態から行動を決める**規則**（「ここでは右へ行く」）。これを良くするのが目的。
- **エピソード**: スタートからゴール（または打ち切り）までの1回の挑戦。
- **リターン**: 1エピソードでの、報酬の**合計**。**これを大きくしたい**。

小さな迷路で、やりとりを実際に動かしてみます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

# 1次元の廊下：マス0〜5。スタートは0、ゴールは5。1歩ごとに報酬 -1（早く着くほど合計が大きい）
class Corridor:
    n = 6
    def reset(self): self.s = 0; return self.s
    def step(self, a):                       # a=+1:右へ, a=-1:左へ
        self.s = int(np.clip(self.s + a, 0, self.n - 1))
        done = self.s == self.n - 1
        return self.s, -1.0, done

def run(policy, rng, max_steps=60):
    env = Corridor(); s = env.reset(); total = 0.0; path = [s]
    for _ in range(max_steps):
        s, r, done = env.step(policy(s, rng)); total += r; path.append(s)
        if done: break
    return total, path

rng = np.random.default_rng(0)
always_right = lambda s, rng: +1
random_walk = lambda s, rng: rng.choice([-1, +1])
print("いつも右へ    : リターン =", run(always_right, rng)[0], " 通った道:", run(always_right, rng)[1])
r, path = run(random_walk, rng); print("でたらめに動く: リターン =", r, f"（{len(path) - 1}歩かかった）")

**出力の見方**: 「いつも右へ」は5歩でゴール、報酬は1歩ごとに −1 なので **リターン −5**（このコースの最高点）。「でたらめに動く」は、行ったり来たりして14歩もかかり、**リターン −14** と大きく悪化します（乱数を固定しているので、毎回同じ数字になります）。
強化学習は、**でたらめな動きから出発して、試行錯誤で「いつも右へ」のような良い方策を、自分で見つける**ことを目指します。このとき、AIは「ゴールは右にある」とは知りません。

### 2-3. 価値 — 「この先どれくらい良いか」の見込み
**状態の価値**は、「ここから最善に進んだとき、リターンがどれくらいになりそうか」の見込みです。**Q値**は「この状態で、この行動をとったときの価値」です。Q値を状態×行動の**表**にしたものが **Q表**。学習とは、**試行錯誤しながら、Q表の数字を直していく**ことです。

In [ ]:
# 廊下の「正解」のQ値（右へ行く／左へ行く）を、後ろから順に計算する（ゴールまでの歩数×-1）
n = 6
V = np.zeros(n)
for s in range(n - 2, -1, -1):
    V[s] = -1 + V[s + 1]                       # 右へ1歩進んで、そこからの価値
Q_right = [-1 + V[min(s + 1, n - 1)] for s in range(n - 1)]
Q_left = [-1 + V[max(s - 1, 0)] for s in range(n - 1)]
print("マス  右へ行く価値  左へ行く価値")
for s in range(n - 1):
    print(f"  {s}      {Q_right[s]:5.0f}        {Q_left[s]:5.0f}   ← {'右' if Q_right[s] > Q_left[s] else '左'}が良い")

**出力の見方**: どのマスでも「右へ行く」ほうが価値が高い（0に近い）ので、方策は「右へ」。ゴールに近いほど価値が高くなります。**学習するAIは、この表を知らない状態から、試行錯誤で作っていきます**。このプロジェクトでは、迷路に限って、この正解を**動的計画法**で別に計算しておき、AIの結果を**正解と比べる**ことができます。

### 2-4. 探索と活用 — いちばん大事なジレンマ
- **活用**: 今まででいちばん良さそうな行動を選ぶ。
- **探索**: まだ試していない行動をあえて選び、もっと良いものを探す。
たとえ話: 行きつけの店だけに通う（活用）と、もっと美味しい店を見逃すかもしれない。新しい店ばかり試す（探索）と、外れも多い。
**ε-greedy**: 確率ε（例: 10%）でランダムに行動（探索）、残りは今の最善（活用）。

### 2-5. 学習のしくみ（Q学習）をざっくり
行動して報酬 r を得て、次の状態 s' に移ったら、**今のQ値を、少しだけ「r + 次の最善の見込み」に近づける**。これを何千回も繰り返す。
- **学習率 α**: 1回でどれだけ近づけるか（大きすぎると不安定）。
- **割引率 γ**: 先の報酬を、どれだけ軽く見るか（γ=0.9なら、1歩先は0.9倍）。
- **深層強化学習（DQN）**: Q表の代わりに**ニューラルネット**を使う。状態が連続値で、表に収まらないとき用。

### 2-6. 報酬設計 — 報酬の言うとおりにしか動かない
AIは、**報酬を最大にする**ことだけをします。人の意図は知りません。報酬の決め方を誤ると、予想外の抜け道（**報酬ハッキング**）を見つけます（例: ゴールに着くより、途中のコインを取り続けるほうが得なら、コインを取り続ける）。

### 2-7. 「ばらつき」と「複数回」 — 評価の約束
強化学習は**乱数でぶれやすい**。1回の結果で判断せず、**乱数の種（シード）を変えて10回**試して、平均と**標準誤差**（平均のあやふやさ）を見ます。差が標準誤差の2倍より小さいなら「差があるとは言えない」と考えます。
また、**学習中の成績**（探索しながら動いた）と、**学習後の成績**（探索をやめて動かした）は、別のものです。

## 3. 用語集（ここに戻ってきてください）

| 用語 | やさしい説明 |
|---|---|
| **強化学習** | 報酬だけを手がかりに、行動を試行錯誤で学ぶ。 |
| **エージェント・環境** | 学ぶ側／行動する相手。ここでは、すべて自作のシミュレータ。 |
| **状態・行動・報酬** | 今の状況／できること／行動の結果の点数。 |
| **方策** | 状態から行動を決める規則。 |
| **エピソード・リターン** | 1回の挑戦／その報酬の合計。 |
| **価値 V・Q値・Q表** | 状態の見込み／状態×行動の見込み／その表。 |
| **動的計画法（価値反復）** | 環境の仕組みを**知っている**とき、最適な方策を厳密に計算する方法。ここでは「正解を知る側」の道具。 |
| **多腕バンディット** | 当たりやすさが違う複数のスロットマシンから、損しないよう選ぶ問題。探索と活用の最小モデル。 |
| **ε-greedy／UCB／トンプソン抽出** | 探索の方法：確率εでランダム／不確かな腕にボーナス／確率分布から選ぶ。 |
| **後悔（regret）** | 最良の選択を続けた場合との差の累積。小さいほど良い。 |
| **Q学習／SARSA** | Q表を更新する2つの方法。Q学習はオフポリシー（最良の行動の価値を学ぶ）、SARSAはオンポリシー（実際にとる行動の価値を学ぶ）。 |
| **学習率 α／割引率 γ／探索率 ε** | 更新の幅／先の報酬の軽視度／ランダムに動く確率。 |
| **報酬ハッキング** | 意図と違う方法で報酬を稼ぐ抜け道。 |
| **報酬整形・楽観的な初期値** | 追加報酬で導く／価値を高めの初期値にして探索を促す。 |
| **DQN・経験再生・ターゲットネットワーク** | Q表をネットに置き換えた手法／過去の経験をためてランダムに使う／目標の計算用のゆっくり動くコピー。 |
| **離散化** | 連続の状態を区切って有限個にすること。 |
| **シード・標準誤差** | 乱数の種／平均のあやふやさ。 |
| **開発用／最終評価用のシード** | 設定を選ぶ用／最後に1回だけ成績をつける用。重ならない。 |
| **蓄電池の制御** | 夜の安い電気を貯め、昼の高い時間に使って電気代を下げる問題。 |

## 4. このプロジェクトの地図（読む順）

| Notebook | 問い | 一言 |
|---|---|---|
| 00 環境と基準 | 迷路で、ランダム／最適はどれくらい？ | 正解を知る準備 |
| 01 バンディット | 探索と活用の最小モデル | 賢く探索すると後悔が減る |
| 02 Q学習・SARSA | 迷路・崖を試行錯誤で攻略 | 学習中と学習後で成績が違う |
| 03 報酬設計 | 報酬を誤るとどうなる？ | 報酬ハッキング |
| 04 DQN | 連続の状態をネットで | 不安定。簡単な方法に負けることも |
| 05 蓄電池 | 実問題に近い設定 | ルールで十分なことも多い |
| 06 評価・最終 | 結果をどこまで信じる？ | 約束を守ると見えるもの |

## 5. 結果を読むときの心得（5つ）
1. **ランダムと最適（正解）の間のどこにいるかを見る。**
2. **学習中の成績と、学習後の成績を区別する。**
3. **複数のシードの平均と、ばらつきを見る。**
4. **ルールなどの簡単な基準と比べる。**
5. **報酬が意図どおりの行動を生んでいるか、疑う。**

## 6. よくある疑問
- **Q. 強化学習は万能？** A. いいえ。簡単なルールで十分なことも多く、学習に膨大な試行が必要なことも多い。ここで数字で確かめます。
- **Q. なぜ自作の環境？** A. 仕組みを理解するためと、正解（最適方策）を計算して比べるためです。
- **Q. 数式が出てきたら？** A. 「出力の見方」の文章だけ追えば大丈夫です。

次は **00_environment** へ進みましょう。